<a href="https://colab.research.google.com/github/Zeldano118/QPon_NLP_PBA/blob/main/notebooks/01_scraping_tokenization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# QPon Review Scraping + NLP Tokenization

| | |
|---|---|
| **App** | [QPon](https://play.google.com/store/apps/details?id=com.qpon.platform) (`com.qpon.platform`) |
| **Author** | Zeldano Shan Oeffie (5026231118) |
| **Course** | Natural Language Processing |

This notebook covers data collection and initial NLP exploration for the QPon sentiment analysis project:
1. Scrape all QPon reviews from Google Play
2. Tokenize English text (Gutenberg) and Indonesian text (IndoNLU) as baselines
3. Tokenize, stem, and remove stopwords from QPon reviews
4. Save raw data for preprocessing in the next notebook

In [ ]:
!pip install google-play-scraper PySastrawi nltk datasets -q

In [ ]:
import pandas as pd
import numpy as np
from google_play_scraper import reviews_all, Sort
from google_play_scraper import app as app_info
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from datasets import load_dataset
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from collections import Counter
import time, os

nltk.download('gutenberg', quiet=True)
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)

---
## A. Scraping QPon Reviews

In [ ]:
info = app_info('com.qpon.platform', lang='id', country='id')

print(f"App        : {info['title']}")
print(f"Developer  : {info['developer']}")
print(f"Rating     : {info['score']:.2f}  ({info['reviews']:,} reviews)")
print(f"Installs   : {info['installs']}")
print(f"Version    : {info['version']}")

In [ ]:
print('Scraping all QPon reviews — this takes a few minutes...\n')
t0 = time.time()

qpon_reviews = reviews_all(
    'com.qpon.platform',
    lang='id',
    country='id',
    sort=Sort.NEWEST,
    sleep_milliseconds=100
)

print(f'Done — {len(qpon_reviews):,} reviews in {time.time()-t0:.0f}s')

In [ ]:
df = pd.DataFrame(qpon_reviews)
print(f'{len(df):,} reviews, {df.shape[1]} columns')
print(f'Columns: {df.columns.tolist()}\n')
df.dtypes

In [ ]:
df.to_csv('qpon_reviews_raw.csv', index=False)
print(f'Saved — {os.path.getsize("qpon_reviews_raw.csv")/1024:.0f} KB')

In [ ]:
# quick look at the data
for _, r in df.head(5).iterrows():
    stars = '\u2B50' * r['score']
    print(f"{stars}  {r['userName']}")
    print(f"   {str(r['content'])[:150]}")
    print()

In [ ]:
print('Rating distribution:\n')
dist = df['score'].value_counts().sort_index()
for score, n in dist.items():
    pct = n / len(df) * 100
    bar = '\u2588' * int(pct / 2)
    print(f'  {score}\u2B50  {n:>5,}  ({pct:4.1f}%)  {bar}')
print(f'\nAverage: {df["score"].mean():.2f}')

In [ ]:
for score in range(1, 6):
    subset = df[df['score'] == score]['content'].dropna()
    if len(subset) == 0: continue
    print(f"\n{'='*50}")
    print(f"Rating {score}/5 — {len(subset):,} reviews")
    print(f"{'='*50}")
    for txt in subset.head(2):
        print(f"  {str(txt)[:180]}")

---
## B. Baseline — English Tokenization (Gutenberg)
Quick tokenization on formal English text as a baseline before moving to Indonesian.

In [ ]:
text_en = nltk.corpus.gutenberg.raw('austen-emma.txt')[:1000]
tokens_en = word_tokenize(text_en)

print('First 500 chars:')
print(text_en[:500])
print(f'\n→ {len(tokens_en)} tokens, {len(set(tokens_en))} unique')

---
## C. Baseline — Indonesian Tokenization (IndoNLU SMSA)
Tokenizing Indonesian social media text from the [IndoNLU SMSA](https://huggingface.co/datasets/kornwtp/indonlu-smsa) dataset. Closer to QPon reviews in style — short, informal, lots of slang.

In [ ]:
dataset = load_dataset('kornwtp/indonlu-smsa', split='train')
df_indo = dataset.to_pandas()
print(f'{len(df_indo)} rows, columns: {list(df_indo.columns)}\n')

text_col = df_indo.columns[0]
label_col = df_indo.columns[1]
samples = df_indo[[text_col, label_col]].head(3)

stemmer = StemmerFactory().create_stemmer()

for i, (_, row) in enumerate(samples.iterrows()):
    text = str(row[text_col])
    tokens = word_tokenize(text)
    stemmed = word_tokenize(stemmer.stem(text))

    print(f'Sample {i+1} (label={row[label_col]}):')
    print(f'  Original : {text}')
    print(f'  Tokens   : {tokens}')
    print(f'  Stemmed  : {stemmed}')
    print(f'  {len(tokens)} tokens → {len(stemmed)} after stemming\n')

    # save token files
    with open(f'tokens_sample_{i+1}.txt', 'w') as f:
        f.write('\n'.join(tokens))

# top words across samples
all_tok = []
for t in samples[text_col]:
    all_tok.extend(word_tokenize(str(t).lower()))
print('Top 10:', Counter(all_tok).most_common(10))

---
## D. Tokenization on QPon Reviews
Applying the same pipeline to the actual QPon data.

In [ ]:
samples_qpon = df['content'].dropna().head(5).tolist()

for i, rev in enumerate(samples_qpon):
    toks = word_tokenize(str(rev))
    print(f'Review {i+1}:')
    print(f'  Text   : {rev}')
    print(f'  Tokens : {toks}')
    print(f'  Count  : {len(toks)}\n')

# frequency across entire dataset
all_qpon = []
for rev in df['content'].dropna():
    all_qpon.extend(word_tokenize(str(rev).lower()))

freq = Counter(all_qpon)
print(f'Total: {len(all_qpon):,} tokens, {len(set(all_qpon)):,} unique\n')
print('Most frequent:')
for w, c in freq.most_common(15):
    print(f'  {w:20s} {c:>6,}')

In [ ]:
# stemming on frequent long words from QPon
long_w = [w for w in set(all_qpon) if len(w) > 6 and w.isalpha()]
long_w = sorted(long_w, key=lambda w: freq[w], reverse=True)[:12]

print('Stemming demo:\n')
for w in long_w:
    s = stemmer.stem(w)
    mark = ' ←' if s != w else ''
    print(f'  {w:22s} ({freq[w]:,}x)  →  {s}{mark}')

# stopword removal
stop = set(stopwords.words('indonesian'))
print(f'\nStopword removal ({len(stop)} Indonesian stopwords):\n')

for i, rev in enumerate(samples_qpon[:3]):
    toks = word_tokenize(str(rev).lower())
    clean = [w for w in toks if w not in stop and w.isalpha()]
    print(f'Review {i+1}:')
    print(f'  Before : {toks}')
    print(f'  After  : {clean}')
    print(f'  Removed {len(toks)-len(clean)} tokens\n')

---
## Summary

**Data:** Scraped 4,638 QPon reviews from Google Play (`com.qpon.platform`, Indonesian). Saved to `qpon_reviews_raw.csv`.

**Tokenization baselines:**
- English (Gutenberg) — formal, well-structured text
- Indonesian (IndoNLU SMSA) — informal social media, closer to app reviews
- QPon reviews — real project data with slang, code-switching, typos

**Key observations:**
- Rating distribution is heavily polarized: ~44% give 1★, ~44% give 5★
- Indonesian affixation (me-, ber-, di-, -kan, -an) is handled well by Sastrawi stemmer
- Stopword removal significantly reduces noise in short review texts